# A. Introduction

Ce notebook présente une solution pour la capture de la provenance des données dans les pipelines de préparation de données. La capture de la provenance est essentielle pour comprendre comment les enregistrements dans un jeu de données résultant d’une transformation ou d’une opération complexe sont reliés à leurs origines dans les jeux de données d’entrée.

### Objectifs

L’objectif principal est de développer une classe Python, appelée `TensorProv`, capable de :

1. Capturer la provenance des données pour trois types d’opérations courantes.
2. Représenter cette provenance à l’aide de tenseurs binaires creux (_sparse tensors_) pour assurer une gestion mémoire et un traitement efficace.
3. Tester et évaluer les performances des approches implémentées, en mesurant notamment les temps d’exécution et l’utilisation des ressources.

### Définition des Méthodes de Capture de Provenance

Deux approches principales sont explorées pour capturer la provenance des données :

1. **Méthode par hashing**  
   Cette méthode consiste à attribuer une clé unique à chaque enregistrement des jeux de données d’entrée en utilisant une fonction de hachage. Ces clés servent ensuite à identifier les correspondances entre les enregistrements des jeux de données de sortie et d’entrée. Bien que cette approche soit efficace pour tracer les liens entre les données, elle peut être coûteuse en termes de calcul pour les grands volumes de données.

2. **Méthode par ajout d’ID**  
   Une colonne supplémentaire contenant des identifiants uniques est ajoutée aux jeux de données d’entrée. Ces identifiants permettent de tracer directement chaque enregistrement des jeux d’entrée vers le jeu de données de sortie. Cette méthode est plus légère en termes de calcul mais requiert des modifications du schéma des données.


### Définition des Opérations Clés

#### **1. Data Transformation**
Les transformations de données consistent à modifier les valeurs spécifiques des attributs d’un jeu de données sans en changer la structure ni le nombre d’enregistrements. Ces transformations peuvent inclure des opérations telles que :
- La **normalisation**, qui ajuste les valeurs pour les ramener dans une échelle spécifique.
- La **discrétisation**, qui divise les valeurs continues en intervalles discrets.
- La **binarisation**, qui convertit les valeurs en données binaires (par exemple, 1 ou 0).

#### **2. Vertical Data Reduction**
Les opérations de réduction verticale consistent à supprimer des colonnes (ou attributs) d’un jeu de données. Cela peut inclure :
- **Feature Selection (sélection des variables)** : Identifier et conserver uniquement les colonnes pertinentes pour notre analyse.
- **Drop Columns (suppression des colonnes)** : Retirer des colonnes inutiles ou redondantes.

Le résultat est un jeu de données avec moins d’attributs mais contenant toujours le même nombre d’enregistrements.

#### **3. Horizontal Data Reduction**
La réduction horizontale vise à filtrer les lignes d’un jeu de données. Cela inclurs :
- **Filtrage** : Supprimer les lignes qui ne remplissent pas certaines conditions (par exemple, supprimer les lignes avec des valeurs manquantes ou aberrantes).
- **Undersampling (sous-échantillonnage)** : Réduire le nombre de lignes pour équilibrer les classes dans un problème de classification.
- **Row Deletion (suppression des lignes)** : Retirer explicitement certaines lignes.

Le résultat est un sous-ensemble des enregistrements d’origine.

#### **4. Vertical Data Augmentation**
L’augmentation verticale modifie le schéma d’un jeu de données sans en changer le nombre de lignes. Cela inclut :
- **Space Transformation** : Projeter les données dans un nouvel espace de caractéristiques (par exemple, utiliser une transformation PCA).
- **String Indexer** : Convertir les valeurs catégoriques en indices numériques.
- **One-Hot Encoding** : Créer des colonnes binaires pour représenter des catégories.

#### **5. Horizontal Data Augmentation**
L’augmentation horizontale génère de nouvelles lignes dans un jeu de données. Les opérations typiques incluent :
- **Oversampling (suréchantillonnage)** : Ajouter des duplications ou interpolations pour équilibrer les classes dans les données.
- **Instance Generation** : Générer artificiellement des enregistrements supplémentaires à partir des données existantes.

#### **6. Join (Fusion des Données)**
L’opération de jointure combine deux jeux de données en fonction d’une ou plusieurs colonnes communes. Cela inclut des types de jointures tels que :
- **Inner Join** : Conserver uniquement les enregistrements correspondants dans les deux jeux.
- **Left/Right Join** : Conserver tous les enregistrements d’un jeu de données et les correspondances dans l’autre.
- **Full Outer Join** : Conserver tous les enregistrements des deux jeux, en remplissant les valeurs manquantes avec `NULL`.

#### **7. Append (Concaténation)**
L’opération d’append ajoute les enregistrements d’un jeu de données à un autre. Contrairement à la jointure, les colonnes des deux jeux n’ont pas besoin d’être identiques :
- Les colonnes absentes sont complétées par des valeurs `NULL`.
- Les enregistrements du second jeu sont ajoutés à la fin du premier.

Cette opération est souvent utilisée pour assembler des ensembles de données qui partagent une structure similaire ou pour consolider des données provenant de différentes sources.


# B. Implémentation de la classe TensorProv

### 1. Implémentation de classe TensorProv


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import (coo_matrix)
import hashlib
import time

class TensorProv:

    def __init__(self, function, method='hash'):
        self.function = function
        self.function_name = self.function.__name__
        self.method = method

    def __call__(self, *args, **kwargs):

        nb_df = TensorProv.count_dataframes(**kwargs)
        if nb_df == 1:
            if self.method == 'ids':
                return self.mono_df_ids_provenance(*args, **kwargs)
            else:
                return self.mono_df_hash_provenance(*args, **kwargs)
        elif nb_df == 2:
            if self.method == 'ids':
                return self.multi_df_ids_provenance(*args, **kwargs)
            else:
                return self.multi_df_hash_provenance(*args, **kwargs)
        else:
            raise ValueError(f"Cannot capture provenance for more than 2 dataframes.")

    @staticmethod
    def count_dataframes(**kwargs):
        return sum(1 for value in kwargs.values() if isinstance(value, pd.DataFrame))

    @staticmethod
    def hash_row_content(row: pd.Series) -> str:
        row_str = row.to_json(date_format="iso", orient="columns")
        return hashlib.md5(row_str.encode("utf-8")).hexdigest()

    def check_call_args(self, *args, **kwargs):

        if "reduced_columns" in kwargs:
            reduced_columns = kwargs["reduced_columns"]
            if isinstance(reduced_columns, list):
                if self.method == "hash" and "_hash_" not in reduced_columns:
                    reduced_columns.append("_hash_")
                if self.method == "ids" and "_id_" not in reduced_columns:
                    reduced_columns.append("_id_")

        return args, kwargs

    def mono_df_hash_provenance(self, *args, **kwargs):

        args, kwargs = TensorProv.check_call_args(self, *args, **kwargs)

        df = kwargs.get('df')
        if df is None:
            raise ValueError(f"Call argument 'df' is required.")

        df["_hash_"] = df.apply(TensorProv.hash_row_content, axis=1)
        result_df = self.function(**kwargs).copy()

        start_time = time.time()
        n:int = len(df)  # total rows in original df
        m: int = len(result_df)  # total rows in filtered df

        hash_to_position = {
            h: i for i, h in enumerate(df["_hash_"].values)
        }

        row_positions = np.arange(m, dtype=np.int32)
        col_positions = np.empty(m, dtype=np.int32)
        data = np.ones(m, dtype=np.int8)

        # For each row i in result_df, find the original row position via the hash
        for i in range(m):
            row_hash = result_df.iloc[i]["_hash_"]
            if row_hash not in hash_to_position:
                raise ValueError(f"Hash {row_hash} in result_df not found in original_df. "
                                 f"Possible data mismatch or missing row?")
            orig_pos = hash_to_position[row_hash]
            col_positions[i] = orig_pos

        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),
            shape=(m, n)
        )
        end_time = time.time()

        return df, result_df, (end_time-start_time), provenance_matrix

    def mono_df_ids_provenance(self, *args, **kwargs):

        args, kwargs = TensorProv.check_call_args(self, *args, **kwargs)

        df = kwargs.get('df')
        if df is None:
            raise ValueError(f"Call argument 'df' is required.")

        df["_id_"] = df.index
        result_df = self.function(**kwargs).copy()

        start_time = time.time()
        n = len(df)
        m = len(result_df)

        orig_indices = result_df["_id_"].to_numpy()
        row_positions = np.arange(m)
        col_positions = orig_indices

        # The data array simply holds ones for each match
        data = np.ones(m, dtype=np.int8)

        # Build the sparse COO matrix
        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),
            shape=(m,n)
        )
        end_time = time.time()

        return df, result_df, (end_time - start_time), provenance_matrix

    def multi_df_hash_provenance(self, *args, **kwargs):
        args, kwargs = TensorProv.check_call_args(self, *args, **kwargs)

        df1 = kwargs.get('df1')
        if df1 is None:
            raise ValueError(f"Call argument 'df1' is required.")

        df2 = kwargs.get('df2')
        if df2 is None:
            raise ValueError(f"Call argument 'df2' is required.")

        df1["_hash_1"] = df1.apply(TensorProv.hash_row_content, axis=1)
        df2["_hash_2"] = df2.apply(TensorProv.hash_row_content, axis=1)

        result_df = self.function(**kwargs).copy()

        start_time = time.time()

        n1 = len(df1)
        n2 = len(df2)
        m = len(result_df)

        t = []

        for i in range(m):
            hash1 = result_df.iloc[i].get("_hash_1", None)
            hash2 = result_df.iloc[i].get("_hash_2", None)

            row_positions = []
            col_positions = []
            data = []

            row_idx = df1[df1["_hash_1"] == hash1].index[0]
            row_positions.append(row_idx)
            row_idx = df2[df2["_hash_2"] == hash2].index[0]
            col_positions.append(row_idx)
            data.append(i)

            provenance_matrix = coo_matrix(
                (data, (row_positions, col_positions)),
                shape=(n1, n2)
            )
            t.append(provenance_matrix)

        end_time = time.time()
        elapsed_time = end_time - start_time

        return df1, df2, result_df, elapsed_time, t

    def multi_df_ids_provenance(self, *args, **kwargs):

        args, kwargs = TensorProv.check_call_args(self, *args, **kwargs)

        df1 = kwargs.get('df1')
        if df1 is None:
            raise ValueError(f"Call argument 'df1' is required.")

        df2 = kwargs.get('df2')
        if df2 is None:
            raise ValueError(f"Call argument 'df2' is required.")

        df1["_id_1"] = np.arange(len(df1))
        df2["_id_2"] = np.arange(len(df2))

        result_df = self.function(**kwargs).copy()

        start_time = time.time()

        n1 = len(df1)
        n2 = len(df2)
        m = len(result_df)


        t = []
        for i in range(m):

            id1 = result_df.iloc[i].get("_id_1", np.nan)
            id2 = result_df.iloc[i].get("_id_2", np.nan)

            row_positions = []
            row_positions.append(int(id1))
            col_positions = []
            col_positions.append(int(id2))
            data = [i]

            # Create a sparse matrix for the i-th row in T
            provenance_matrix = coo_matrix(
                (data, (row_positions, col_positions)),
                shape=(n1,n2)
            )
            t.append(provenance_matrix)

        end_time = time.time()
        elapsed_time = end_time - start_time

        return df1, df2, result_df, elapsed_time, t


### 2. Implémentation de classe et methodes utilitaires

In [90]:
import random, string
import pandas as pd
import numpy as np
from typing import Literal
from imblearn.over_sampling import SMOTE


######### ---------------------------------------------------------------------------------------------------- #########
######### Méthodes de transformation que nous utiliserons dans la cadre des captures de provenance             #########
######### ---------------------------------------------------------------------------------------------------- #########


def query_function(df: pd.DataFrame, condition: str):
    return df.query(condition)


def reduce_function(df: pd.DataFrame, reduced_columns: list):
    return df[reduced_columns]


def hda_function(df: pd.DataFrame):
    x = df[['age']]
    y = df['city']
    x_resampled, y_resampled = SMOTE(k_neighbors=1).fit_resample(X, y)
    return pd.concat([
        pd.DataFrame(x_resampled, columns=x.columns),  # Caractéristiques augmentées
        pd.Series(y_resampled, name='city')  # Labels augmentés
    ],
        axis=1
    )
######### ---------------------------------------------------------------------------------------------------- #########
######### Méthodes utilitaires pour la génération de datafarme de tests                                        #########
######### ---------------------------------------------------------------------------------------------------- #########
def random_string(length=8):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def generate_large_df(n_persons=1000):

    # Generate the person DataFrame
    persons_df = pd.DataFrame({
        "name": [random_string() for _ in range(n_persons)],
        "age": np.random.randint(18, 70, size=n_persons),
        "city": np.random.choice(["NY", "SF", "LA", "Berlin", "London"], size=n_persons)
    })

    hobbies_list = []
    for name in persons_df["name"]:
        n_hobbies = random.randint(0, 3)  # Randomly choose between 0 and 3 hobbies
        hobbies = np.random.choice(
            ["Reading", "Painting", "Cycling", "Cooking", "Gardening", "Hiking"],
            size=n_hobbies,
            replace=False
        )
        for hobby in hobbies:
            hobbies_list.append({"name": name, "hobbies": hobby})
    hobbies_df = pd.DataFrame(hobbies_list)

    return persons_df, hobbies_df

######### ---------------------------------------------------------------------------------------------------- #########
######### Mthode utilitaire permettant l'exceution des test et l'affichage d'n tableau récapitulatif           #########
######### ---------------------------------------------------------------------------------------------------- #########

def execute_test_and_log(df_counts: list, test_function):
    test_logs = []
    for count in df_counts:
        persons_df, hobbies_df = generate_large_df(n_persons=count)
        print(f"Running {test_function.__name__} tests with {count} rows")
        hash_tensor_filter = TensorProv(function= test_function, method='hash')
        df1, df2, result_df, execution_time, provenance_matrix_list  = hash_tensor_filter(df1=persons_df, df2=hobbies_df, on_merge="name", how_merge="inner")
        test_logs.append({
            "Test Name": f"Test {test_function.__name__} with {count} rows",
            "Capture Method": "Hash",
            "Original (Person) Rows": len(persons_df),
            "Original (Hobbies) Rows": len(hobbies_df), 
            "Transformation Function": test_function.__name__,
            "Result Rows": len(result_df),
            "Execution Time (s)": execution_time
        })
        ids_tensor_filter = TensorProv(function= test_function, method='ids')
        df1, df2, result_df, execution_time, provenance_matrix_list  = ids_tensor_filter(df1=persons_df, df2=hobbies_df, on_merge="name", how_merge="inner")
        test_logs.append({
            "Test Name": f"Test {test_function.__name__} with {count} rows",
            "Capture Method": "Ids",
            "Original (Person) Rows": len(persons_df),
            "Original (Hobbies) Rows": len(hobbies_df), 
            "Transformation Function": test_function.__name__,
            "Result Rows": len(result_df),
            "Execution Time (s)": execution_time
        })    

    df_logs = pd.DataFrame(test_logs)
    print("\nTest Summary:")
    print(df_logs.to_string(index=False))   

# C. Test de différentes transformation

### 1. Opération de fusion (merge)

In [ ]:

def merge_func(df1: pd.DataFrame, df2: pd.DataFrame, on_merge: str, how_merge: Literal["left", "right", "inner", "outer", "cross"] = "inner") -> pd.DataFrame:
    return df1.merge(df2, on=on_merge, how=how_merge, suffixes=('_x', '_y'))

df_counts = [10,100,1000,10000, 25000, 50000]
execute_test_and_log(df_counts, merge_func)


Running merge_func tests with 10 rows
Running merge_func tests with 100 rows
Running merge_func tests with 1000 rows
Running merge_func tests with 10000 rows
Running merge_func tests with 25000 rows
Running merge_func tests with 50000 rows
